In [1]:
import os
import glob
import tgt
import pandas as pd
import pickle
import tqdm
import re
import os
import warnings
warnings.simplefilter("ignore")


import sys
sys.path.insert(0, "../../..")

from utils import data_processing
DATASET_PATH = "/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16/"

In [2]:
tgrd_fs = glob.glob(os.path.join(DATASET_PATH, "sub*/alignment/*.TextGrid"))
def filter_files(tgrd_fs):
    subject_info = pd.read_excel(os.path.join(DATASET_PATH, "Subjects.xlsx"))

    def get_subject_id(tgrd_f):
        subject_id = tgrd_f.split("/")[5]
        if subject_id=="usc":
            print(tgrd_f)
        return subject_id
    def is_native(subject_id):
        return subject_info[subject_info["Subject ID"] == subject_id]["Origin"].values[0] == "USA"
    
    filtered_fs = []
    for f in tgrd_fs:
        subject_id = get_subject_id(f)
        if is_native(subject_id):
            filtered_fs.append(f)
    return filtered_fs

tgrd_fs = filter_files(tgrd_fs)

def read_tgrd(tgrd_f):
    try:
        tgrd = tgt.read_textgrid(tgrd_f)
    except UnicodeError:
        tgrd = tgt.read_textgrid(tgrd_f, encoding="utf-16")
    return tgrd

tgrds = [(read_tgrd(f), f) for f in tgrd_fs]

In [3]:
preferred_ipas_f = "../../../phone_based/pronunciation_counts/preferred_ipas_ae.pkl"

In [4]:
all_mp4s = glob.glob("/Volumes/Transcend/USC-75/sub*/2drt/video/*_video.mp4")
all_mats = glob.glob("/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16/sub*/track/*_track.mat")

def get_wav_path(tgrd_f):

    def normalize_path(path: str) -> str: return re.sub(r"_(\d+)_", "_", path).lower()
    
    identifier = os.path.basename(tgrd_f).replace(".TextGrid", "")
    target_mp4 = os.path.join(f"/Volumes/Transcend/USC-75/{[i for i in identifier.split('_') if i.startswith('sub')][0]}/2drt/video/", identifier.replace("_text", "_video.mp4"))
    
    for mp4 in all_mp4s:
        if normalize_path(mp4) == normalize_path(target_mp4):
            return mp4.replace(".mp4", ".wav")
    return None

def get_mat_path(tgrd_f):
    identifier = os.path.basename(tgrd_f).replace(".TextGrid", "")
    wav = get_wav_path(tgrd_f)
    def get_subject_id(tgrd_f):
        subject_id = [i for i in tgrd_f.split("/") if i.startswith("sub")][0]
        return subject_id
    def normalize_path(path: str) -> str:
        import re
        # remove underscore + digits + underscore (but keep the underscores collapsed)
        return re.sub(r"_(\d+)_", "_", path).lower()
    target_mat = os.path.join(f"/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16/{get_subject_id(tgrd_f)}/track/", os.path.basename(wav).replace("_video.wav", "_track.mat"))

    for mat in all_mats:
        if normalize_path(mat) == normalize_path(target_mat):
            return mat
    return None

In [5]:
dfs_final = []

for tgrd, tgrd_f in tqdm.tqdm(tgrds):

    df_final = data_processing.alignment_to_dataframe(
        tgrd,
        lang="ae",
        preferred_ipas_f=preferred_ipas_f,
        unit_type="syllable"
    )

    art_f = get_mat_path(tgrd_f)
    ac_f = get_wav_path(tgrd_f)
    df_final["art_f"] = art_f
    df_final["ac_f"] = ac_f
    dfs_final.append(df_final)

  0%|          | 0/192 [00:00<?, ?it/s]

100%|██████████| 192/192 [03:29<00:00,  1.09s/it]


In [6]:
df = pd.concat(dfs_final, ignore_index=True)

In [7]:
df.to_csv("datasets/processed_aemri75anno.csv", index=False)